In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
data={
    "YEAR": [],
    "MONTH_NUM": [],
    "MONTH_MON": [],
    "FLT_DATE": [],
    "APT_ICAO": [],
    "APT_NAME": [],
    "STATE_NAME": [],
    "FLT_DEP_1": [],
    "FLT_ARR_1": [],
    "FLT_TOT_1": [],
    "FLT_DEP_IFR_2": [],
    "FLT_ARR_IFR_2": [],
    "FLT_TOT_IFR_2": []
}
df=pd.read_csv("/content/airport_traffic_2025[1].csv")
print(df.head())

In [ ]:
print(df.isna().sum())
df["FLT_DEP_IFR_2"]=df["FLT_DEP_IFR_2"].fillna(0)
df["FLT_ARR_IFR_2"]=df["FLT_ARR_IFR_2"].fillna(0)
df["FLT_TOT_IFR_2"]=df["FLT_TOT_IFR_2"].fillna(0)
print(df.head())

In [ ]:
df["FLT_DATE"]=pd.to_datetime(df["FLT_DATE"])

In [ ]:
df['total_attendu'] = df['FLT_DEP_1'] + df['FLT_ARR_1']
df['anomalie'] = df['FLT_TOT_1'] != df['total_attendu']
df.loc[df['anomalie'], 'FLT_TOT_1'] = df['total_attendu']
print(df.head())

In [ ]:
df['trafic_VFR'] = df['FLT_TOT_1'] - df['FLT_TOT_IFR_2']
df.loc[df['trafic_VFR'] < 0, 'trafic_VFR'] = 0
print(df.head())

In [ ]:
df['ratio_IFR'] = (df['FLT_TOT_IFR_2'] / df['FLT_TOT_1']) * 100
print(df.head())

In [ ]:
df['jour_semaine'] = df['FLT_DATE'].dt.day_name()
df['num_semaine'] = df['FLT_DATE'].dt.isocalendar().week
print(df.head())

In [ ]:
def segmenter(vols):
    if vols < 1000:
        return "Petit"
    elif vols < 1200:
        return "Moyen"
    else:
        return "Hub International"
df['segment'] = df['FLT_TOT_1'].apply(segmenter)
print(df.head())

In [2]:
top10_hubs = df.groupby("APT_NAME")["FLT_TOT_1"].sum().nlargest(10)
print("Top 10 hubs par trafic total :")
print(top10_hubs)

NameError: name 'df' is not defined

In [ ]:
top_countries = df.groupby("STATE_NAME")["FLT_TOT_1"].sum().sort_values(ascending=False)
print("\nPays les plus actifs en volume de vols :")
print(top_countries)

In [ ]:
df['ratio_arr_dep'] = df['FLT_ARR_1'] / df['FLT_DEP_1']
print("Ratio Arrivées/Départs par aéroport :")
print(df[['APT_ICAO', 'ratio_arr_dep']])

In [ ]:
top5_hotspots = df.groupby("APT_ICAO")["FLT_TOT_IFR_2"].sum().nlargest(5)
print("\nTop 5 hotspots IFR :")
print(top5_hotspots)

In [ ]:
df['month'] = df['FLT_DATE'].dt.month
monthly_growth = df.groupby('month')["FLT_TOT_IFR_2"].sum().pct_change().fillna(0)
print("\nCroissance mensuelle du trafic IFR :")
print(monthly_growth)

In [ ]:
jour_record = df.loc[df['FLT_TOT_1'].idxmax(), ['FLT_DATE', 'FLT_TOT_1']]
print("Jour record de 2025:\n", jour_record)

In [ ]:
df['month'] = df['FLT_DATE'].dt.month
monthly_growth = df.groupby('month')["FLT_TOT_1"].sum().pct_change().fillna(0)
mois_max_growth = monthly_growth.idxmax()
print("Mois de plus forte croissance :", mois_max_growth)

In [ ]:
top_country = df.groupby("STATE_NAME")["FLT_TOT_1"].sum().idxmax()
print("Réseau national le plus dynamique :", top_country)

In [ ]:
stability = df.groupby("APT_ICAO")["FLT_TOT_1"].std()
stable_airport = stability.idxmin()
print("Aéroport le plus stable :", stable_airport)

In [ ]:
monthly_totals = df.groupby("month")["FLT_TOT_1"].sum()
plt.figure(figsize=(10,6))
plt.plot(monthly_totals.index, monthly_totals.values, marker="o", linestyle="-", color="green")
plt.title("Évolution mensuelle du trafic total")
plt.xlabel("Mois")
plt.ylabel("Nombre total de vols")
plt.grid(True)
plt.show()

In [ ]:
total_IFR = df["FLT_TOT_IFR_2"].sum()
total_VFR = df["FLT_TOT_1"].sum() - total_IFR
plt.figure(figsize=(6,6))
plt.pie([total_IFR, total_VFR], labels=["IFR", "VFR"], autopct="%1.1f%%", colors=["orange","lightblue"])
plt.title("Proportion globale IFR vs VFR")
plt.show()